## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [0]:
import tensorflow as tf

In [0]:
tf.enable_eager_execution()


In [0]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default


### Collect Data

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [0]:
import pandas as pd
import numpy as np

In [0]:
data = pd.read_csv('/content/drive/My Drive/Neural Network/Lab/prices.csv')

### Check all columns in the dataset

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 851264 entries, 0 to 851263
Data columns (total 7 columns):
date      851264 non-null object
symbol    851264 non-null object
open      851264 non-null float64
close     851264 non-null float64
low       851264 non-null float64
high      851264 non-null float64
volume    851264 non-null float64
dtypes: float64(5), object(2)
memory usage: 45.5+ MB


### Drop columns `date` and  `symbol`

In [7]:
# drop date and symbol column as it is useless for the model
if 'date' in data.columns:
    data.drop(['date'], axis=1, inplace = True)
if 'symbol' in data.columns:
    data.drop(['symbol'], axis=1, inplace = True)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 851264 entries, 0 to 851263
Data columns (total 5 columns):
open      851264 non-null float64
close     851264 non-null float64
low       851264 non-null float64
high      851264 non-null float64
volume    851264 non-null float64
dtypes: float64(5)
memory usage: 32.5 MB


In [8]:
data.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [0]:
df = data.head(1000)


In [10]:
df['volume'] = df.apply(lambda row: row['volume'] /1000000, axis=1)

df.head(10)

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """Entry point for launching an IPython kernel.


,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2.1636
1,125.239998,119.980003,119.940002,125.540001,2.3864
2,116.379997,114.949997,114.930000,119.739998,2.4895
3,115.480003,116.620003,113.500000,117.440002,2.0063
4,117.010002,114.970001,114.089996,117.330002,1.4086
5,115.510002,115.550003,114.500000,116.059998,1.0980
6,116.459999,112.849998,112.589996,117.070000,0.9496
7,113.510002,114.379997,110.050003,115.029999,0.7853
8,113.330002,112.529999,111.919998,114.879997,1.0937
9,113.660004,110.379997,109.870003,115.870003,1.5235


### Divide the data into train and test sets

In [0]:

# Prepare X and Y
X = df.drop("volume", axis=1)
y = df["volume"]

In [0]:
from sklearn.model_selection import train_test_split

# split into 67% for train and 33% for test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=7)

#### Convert Training and Test Data to numpy float32 arrays


In [0]:
X_train = X_train.to_numpy(dtype="float32")
X_test = X_test.to_numpy(dtype="float32")

In [23]:
type(y_train)

pandas.core.series.Series

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [0]:
from sklearn.preprocessing import Normalizer

transformer = Normalizer().fit(X_train)
X_train = transformer.transform(X_train)
X_test = transformer.transform(X_test)

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [0]:
W = tf.Variable(tf.random_normal(shape=[4,1]), name="Weights")
b = tf.Variable(tf.zeros(shape=[1]),name="Bias")

2.Define a function to calculate prediction

In [0]:
def fn_prediction(x,W,b):
  y_pred = tf.add(tf.matmul(X_train,W),b,name='output')
  return y_pred

3.Loss (Cost) Function [Mean square error]

In [0]:
def fn_loss(y,y_pred):
  loss = tf.reduce_mean(tf.square(np.asarray(y)-y_pred),name='Loss')
  return loss

4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [0]:
def fn_train(x,y,W,b,learn_rate):
  with tf.GradientTape() as t :
     t.watch([W,b])
     y_pred = fn_prediction(x,W,b)
     loss = fn_loss(y,y_pred)
  
  dc_dw, dc_db = t.gradient(loss, [W, b])
  W = W - learn_rate * dc_dw
  b = b - learn_rate * dc_db
  return W,b

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [79]:
for i in range (100):
  W,b = fn_train(X_train,y_train,W,b,0.01)
  y_pred = fn_prediction(X_train,W,b)
  train_loss = fn_loss(y_train,y_pred)
  if i % 5 == 0:
    print('Training loss at step: ', i, ' is ', train_loss)


Training loss at step:  0  is  tf.Tensor(308.74023, shape=(), dtype=float32)
Training loss at step:  5  is  tf.Tensor(300.9423, shape=(), dtype=float32)
Training loss at step:  10  is  tf.Tensor(295.7579, shape=(), dtype=float32)
Training loss at step:  15  is  tf.Tensor(292.31116, shape=(), dtype=float32)
Training loss at step:  20  is  tf.Tensor(290.0196, shape=(), dtype=float32)
Training loss at step:  25  is  tf.Tensor(288.4961, shape=(), dtype=float32)
Training loss at step:  30  is  tf.Tensor(287.48322, shape=(), dtype=float32)
Training loss at step:  35  is  tf.Tensor(286.80978, shape=(), dtype=float32)
Training loss at step:  40  is  tf.Tensor(286.3621, shape=(), dtype=float32)
Training loss at step:  45  is  tf.Tensor(286.06442, shape=(), dtype=float32)
Training loss at step:  50  is  tf.Tensor(285.86655, shape=(), dtype=float32)
Training loss at step:  55  is  tf.Tensor(285.73495, shape=(), dtype=float32)
Training loss at step:  60  is  tf.Tensor(285.6475, shape=(), dtype=flo

### Get the shapes and values of W and b

In [89]:
print("Shape and Values of W")
print(W)

Shape and Values of W
tf.Tensor(
[[0.3855927]
 [1.7564299]
 [1.2894689]
 [2.846911 ]], shape=(4, 1), dtype=float32)


In [90]:
print("Shape and Values of b")
print(b)

Shape and Values of b
tf.Tensor([2.4699142], shape=(1,), dtype=float32)


### Model Prediction on 1st Examples in Test Dataset

In [87]:
y_pred = fn_prediction(X_test,W,b)
test_loss = fn_loss(y_test,y_pred)
print('Loss in Test set is', test_loss)

Loss in Test set is tf.Tensor(54.13903, shape=(), dtype=float32)


## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

In [0]:
# import data set
df_iris = pd.read_csv( '/content/drive/My Drive/Neural Network/Lab/iris.csv' )

In [114]:
df_iris.head(5)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [0]:
# drop ID column as it is useless for the model
if 'Id' in df_iris.columns:
    df_iris.drop(['Id'], axis=1, inplace = True)

### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

In [0]:
# convert categorical data into numeric data
#from sklearn.preprocessing import LabelEncoder
#le = LabelEncoder()
#df_iris["Species"] = le.fit_transform(df_iris["Species"])
#print(df_iris.groupby('Species').size())

In [0]:
df_iris = pd.get_dummies(df_iris, drop_first = False)

In [126]:
df_iris.head(5)

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species_Iris-setosa,Species_Iris-versicolor,Species_Iris-virginica
0,5.1,3.5,1.4,0.2,1,0,0
1,4.9,3.0,1.4,0.2,1,0,0
2,4.7,3.2,1.3,0.2,1,0,0
3,4.6,3.1,1.5,0.2,1,0,0
4,5.0,3.6,1.4,0.2,1,0,0


In [0]:
from keras.utils import np_utils

In [0]:
#Encoding the output class label (One-Hot Encoding)
y_train=np_utils.to_categorical(ytrain,3)
y_test=np_utils.to_categorical(ytest,3)

### Splitting the data into feature set and target set

In [0]:
# Prepare X and Y
colname = ['Species_Iris-setosa','Species_Iris-versicolor','Species_Iris-virginica']
X = df_iris.drop(colname, axis=1)
Y=df_iris[colname]


###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [132]:
from keras.models import Sequential
from keras.layers.convolutional import Conv1D
from keras.layers.convolutional import MaxPooling1D
from keras.layers import Flatten
from keras.layers import Dense

Using TensorFlow backend.


In [0]:
model = tf.keras.Sequential([
  tf.keras.layers.Dense(10, activation=tf.nn.relu, input_shape=(4,)),  # input shape 4
  tf.keras.layers.Dense(3,activation=tf.nn.softmax) # output shape 3
])

### Model Training 

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.33, random_state=7)

In [0]:
model.compile(loss='categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])

In [154]:
model.fit(X_train, y_train, validation_data=(X_test,y_test), epochs=50, batch_size=10)


Train on 100 samples, validate on 50 samples
Epoch 1/50
100/100 [==============================] - 0s 2ms/sample - loss: 0.9656 - acc: 0.4800 - val_loss: 0.8667 - val_acc: 0.6400
Epoch 2/50
100/100 [==============================] - 0s 358us/sample - loss: 0.7913 - acc: 0.6800 - val_loss: 0.8023 - val_acc: 0.6400
Epoch 3/50
100/100 [==============================] - 0s 397us/sample - loss: 0.7287 - acc: 0.6800 - val_loss: 0.7505 - val_acc: 0.6400
Epoch 4/50
100/100 [==============================] - 0s 375us/sample - loss: 0.6775 - acc: 0.6800 - val_loss: 0.7161 - val_acc: 0.6400
Epoch 5/50
100/100 [==============================] - 0s 369us/sample - loss: 0.6398 - acc: 0.6800 - val_loss: 0.6755 - val_acc: 0.6400
Epoch 6/50
100/100 [==============================] - 0s 333us/sample - loss: 0.6047 - acc: 0.6800 - val_loss: 0.6418 - val_acc: 0.6400
Epoch 7/50
100/100 [==============================] - 0s 338us/sample - loss: 0.5767 - acc: 0.6800 - val_loss: 0.6237 - val_acc: 0.6600
Epoch

### Model Prediction

In [159]:
scores = model.evaluate(X_test,y_test, verbose=0)
print(scores)
print("Prediction Accuracy is", scores[1]*100)

[0.3014664655923843, 0.96]
Prediction Accuracy is 95.99999785423279


### Save the Model

In [0]:
model.save("/content/drive/My Drive/Neural Network/Lab/iris_model.h5")

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?